In [ ]:
# annotated example of extracting non-linear cyclpoint-related activation 

# necessary libraries
%matplotlib inline

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nibabel as nib
from nltools.file_reader import onsets_to_dm
from nltools.stats import regress, zscore, fdr
from nltools.data import Brain_Data, Design_Matrix
from nilearn.plotting import view_img, glass_brain, plot_stat_map
from bids import BIDSLayout, BIDSValidator
import statsmodels.api as sm
from statsmodels.gam.api import GLMGam, CyclicCubicSplines
import multiprocessing as mp
import warnings
warnings.filterwarnings('ignore') 

In [ ]:
data_dir = '/data/e2brain' # replace this with your own data directory
layout = BIDSLayout(data_dir,derivatives=True)
output_dir = os.path.join(data_dir,'derivatives','glm','gam') # and your desired output directory
cyc = pd.read_csv('acc_cyc.csv') # this is a dataframe with participant-level cyclepoint and relevant task performance values

In [ ]:
zstat = 4
mask = 'mean_group_mask_MNI.nii.gz' # mask you want to use
lev1_dir = os.path.join(data_dir,'derivatives','glm','level1') # directory containing your level1 analysis results 
zfiles = []
cyclepoint = []
sbj = []
acc_nf = []
for ix,c in cyc.iterrows():
    zfile = os.path.join(lev1_dir,f'{c.sbj}_task-learn_run-nf.feat','stats',f'zstat{zstat}.nii.gz') # adapt to your file naming convention
    if os.path.exists(zfile):
        zfiles.append(zfile)
        cyclepoint.append(c.cyclepoint)
        sbj.append(c.sbj)
        acc_nf.append(c.NF)

zimg = Brain_Data(zfiles,mask=mask)
nvox = zimg.shape()[1]
cyclepoint = np.array(cyclepoint)
acc_nf = np.array(acc_nf)
zimg.write(os.path.join(output_dir,f'zstat{zstat}.nii.gz'))
pd.DataFrame({'sbj':sbj,'cyclepoint':cyclepoint,'acc_nf':acc_nf}).to_csv(os.path.join(output_dir,f'cyclepoint.csv'),index=False)

In [4]:
pd.DataFrame({'sbj':sbj,'cyclepoint':cyclepoint,'acc_nf':acc_nf}).to_csv(os.path.join(output_dir,f'cyclepoint.csv'),index=False)

In [ ]:
def run_gam(i):
    if i % np.round(nvox/10) == 0 and i > 0:
        print(f'{i}/{nvox} voxels processed')
    df = pd.DataFrame(np.array([zimg.data[:,i],cyclepoint]).T,columns=['z','cyclepoint'])
    cc = CyclicCubicSplines(df['cyclepoint'],df=[10])
    stat = sm.gam.GLMGam.from_formula('z~cyclepoint',data=df,smoother=cc).fit().test_significance(0)
    return([stat.pvalue,stat.statistic[0][0]])

pool = mp.Pool(10)
results = pool.map(run_gam,range(nvox))
results = np.array(results)

pmap = zimg.copy()
pmap.data = np.reshape(1-results[:,0],(1,nvox))
pmap.write(os.path.join(output_dir,f'zstat{zstat}_cyclepoint_pvalue.nii.gz'))
chi2map = zimg.copy()
chi2map.data = np.reshape(results[:,1],(1,nvox))
chi2map.write(os.path.join(output_dir,f'zstat{zstat}_cyclepoint_chi2.nii.gz'))

In [6]:
df = pd.DataFrame(np.array([zimg.data[:,1],cyclepoint]).T,columns=['z','cyclepoint'])
cc = CyclicCubicSplines(df['cyclepoint'],df=[10])
pvalue = np.zeros(nvox)
chi2 = np.zeros(nvox)
for i in range(1):
    df['z'] = zimg.data[:,i]
    res = sm.gam.GLMGam.from_formula('z~cyclepoint',data=df,smoother=cc).fit()
    stat = res.test_significance(0)
    pvalue[i] = stat.pvalue
    chi2[i] = stat.statistic


In [ ]:
res

In [ ]:
cyclepoint